# GRPO: подготовка, проверки, обучение

Спецификация — `docs/grpo.md`. Порядок: (1) слить SFT-адаптер с базой — один раз; (2) запустить vLLM-сервер в его
окружении на карте 1 (терминал); (3) проверки до длинных запусков; (4) ответы SFT на validation — один раз для всех
запусков; (5) запуск обучения или продолжение после сбоя; (6) кривые по журналам — в любой момент, обучение не трогают.

Шаги 1, 3, 4 и 5 занимают карту 0 — между ними лучше перезапускать ядро (освобождение памяти).

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))  # если пакет не установлен через pip install -e .
from presentation_rl.training.common import RLConfig
from presentation_rl.training.vllm_client import VLLMClient

# Википедия: пока она закрыта прокси пространства — через туннель с компьютера автора (scripts/wiki_tunnel,
# docs/server.md); когда откроют белый список — WIKI_PROXY = "" (прямой доступ)
os.environ["WIKI_PROXY"] = "http://127.0.0.1:18080"
config = RLConfig.load(ROOT / "configs" / "algorithms" / "grpo.yaml")
SFT_ADAPTER = "runs/sft/20261107_1913/adapter"  # EXP-015, эпоха 3
print(config.sft_model, config.vllm_url, config.steps, config.max_context)

## 1. Слияние SFT-адаптера с базой (один раз)

ΔW считается во float32, результат — float16 (`docs/grpo.md`, раздел 8). Рядом с весами — `merged_hash.json`
(хеши), его читает манифест каждого запуска. Около 16 ГБ на диске.

In [ ]:
from presentation_rl.training.common import merge_sft

out = ROOT / config.sft_model
if (out / "merged_hash.json").exists():
    print("уже слито:", out)
else:
    info = merge_sft("t-tech/T-lite-it-2.1", "main", ROOT / SFT_ADAPTER, out, device="cuda:0")
    print(json.dumps(info, ensure_ascii=False, indent=1)[:1500])

## 2. Сервер vLLM (в терминале, отдельное окружение, карта 1)

Один раз — окружение (`~/vllm085`):
```bash
python3.11 -m venv ~/vllm085 && source ~/vllm085/bin/activate
pip install vllm==0.8.5 torch==2.6.0 xformers==0.0.29.post2 transformers==4.51.3
```
Запуск (держать открытым, например в `tmux`); путь к модели — абсолютный путь `config.sft_model`:
```bash
source ~/vllm085/bin/activate
VLLM_USE_V1=0 VLLM_ALLOW_RUNTIME_LORA_UPDATING=True CUDA_VISIBLE_DEVICES=1 \
python -m vllm.entrypoints.openai.api_server --model <ПУТЬ>/runs/sft/merged_20261107_1913 \
    --served-model-name sft --dtype float16 --enable-lora --max-lora-rank 32 --max-loras 1 \
    --max-model-len 25600 --gpu-memory-utilization 0.80 --generation-config vllm --port 8000 --seed 0
```
`--gpu-memory-utilization 0.80` оставляет на карте 1 место для bge-m3 (награда). Ячейка ниже проверяет связь.

In [ ]:
client = VLLMClient(config.vllm_url)
print("vLLM", client.version(), client.models())

Проверка Википедии: прямой запрос к API (не из кеша) через `WIKI_PROXY`. Ошибка «Connection refused» — туннель
не запущен на компьютере автора (`tunnel.ps1`) или SSH-сессия попала не на ту машину, где работает ядро.

In [ ]:
import time
from presentation_rl.tools.wiki import WikiClient

wiki = WikiClient(contact=os.environ.get("WIKI_CONTACT", "https://github.com/doshik722"), network_patience=0)
start = time.monotonic()
hits = wiki.search("ru", "фотосинтез")
print(f"WIKI_PROXY={os.environ['WIKI_PROXY']!r}: найдено {len(hits)}, {time.monotonic() - start:.1f} с;",
      [h.title for h in hits[:3]])

## 3. Проверки до длинных запусков

(а) тесты на крошечной модели: log p кусками, нулевой адаптер, градиент ratio, накопление, градиент k2, направление
обновления; (б) память карты 0 — полный шаг на эпизоде с максимальным контекстом и 8000 токенов модели (наивное ядро
внимания выключено); (в) пиковая нагрузка карты 1 — vLLM и bge-m3 вместе; (г) log p vLLM против обучения — по ним
задаётся порог `max_logprob_mismatch` в конфиге; (д) смена адаптеров A → B → A.

In [ ]:
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_rl_torch.py", "tests/test_rl_common.py",
                         "tests/test_environment.py"], cwd=ROOT, capture_output=True, text=True)
print(result.stdout[-3000:], result.stderr[-2000:])

In [ ]:
from presentation_rl.training.checks import memory_check

print(memory_check(config, ROOT))  # ok=False — нехватка памяти: уменьшить max_context в конфиге и повторить

In [ ]:
from presentation_rl.rewards import SentenceEmbedder
from presentation_rl.training.checks import vllm_load_check

embedder = SentenceEmbedder("BAAI/bge-m3", device=config.embedder_device)
print(vllm_load_check(config, client, embedder, parallel=8))

In [ ]:
from presentation_rl.tasks import load_tasks
from presentation_rl.training.checks import adapter_switch_check, logprob_check
from presentation_rl.training.common import episode_prompt, load_policy, mode_task

tasks = {t.id: t for t in load_tasks(ROOT / "data" / "tasks" / "tasks.json", split="train")}
TOPICS = sorted(tasks)[:4]
MODES = ("plain",)  # с доступом к Википедии — ("search", "plain")
model, tokenizer = load_policy(config, ROOT)
print(logprob_check(config, ROOT, client, tasks, TOPICS, model, tokenizer, modes=MODES))
prompt_ids = episode_prompt(tokenizer)(mode_task(tasks[TOPICS[0]], "plain").request)
print(adapter_switch_check(config, ROOT, client, model, tokenizer, prompt_ids))

## 4. Ответы SFT на validation (один раз для всех запусков)

25 тем × 2 режима × 2 ответа с сидами validation; с ними судья сравнивает чекпоинты. Результат —
`runs/grpo/reference/`.

In [ ]:
from presentation_rl.training.grpo import generate_reference

print(generate_reference(config, ROOT, client))

## 5. Обучение

`RUN_DIR = None` — новый запуск (папка `runs/grpo/<дата_время>_seed<сид>`); путь к папке — продолжение после сбоя
с последнего полного состояния (каждые 10 шагов). Сид — `config.seed`: второй запуск — `RLConfig.load(..., seed=1)`.

In [ ]:
from presentation_rl.training.grpo import train

RUN_DIR = None  # например ROOT / "runs/grpo/20261110_1432_seed0" — продолжение
run = train(config, ROOT, run_dir=RUN_DIR, client=client)
print(run)

## 6. Кривые

Читает журналы запуска (`steps.jsonl`, `validation.jsonl`, `judge.jsonl`, если судья уже работает); отменённые
продолжением шаги не учитываются. Можно запускать во время обучения.

In [ ]:
import matplotlib.pyplot as plt
from presentation_rl.training.common import read_journal

RUN = sorted((ROOT / config.runs_dir).glob("*_seed*"))[-1]  # или конкретная папка
steps = [r for r in read_journal(RUN / "steps.jsonl") if "step" in r]
val = [r for r in read_journal(RUN / "validation.jsonl") if "step" in r]
judge = [r for r in read_journal(RUN / "judge.jsonl") if "step" in r]
panels = [("награда (обучение)", [(r["step"], r.get("search_reward")) for r in steps], [(r["step"], r.get("plain_reward")) for r in steps]),
          ("награда (validation)", [(r["step"], r.get("search_reward")) for r in val], [(r["step"], r.get("plain_reward")) for r in val]),
          ("KL к опорной (k3)", [(r["step"], r.get("kl_ref_k3")) for r in steps], None),
          ("энтропия", [(r["step"], r.get("entropy")) for r in steps], None),
          ("доля поиска / опора на прочитанное", [(r["step"], r.get("search_share")) for r in steps], [(r["step"], r.get("grounding")) for r in steps]),
          ("доля побед над SFT / согласие", [(r["step"], r.get("win_rate")) for r in judge], [(r["step"], r.get("agreement")) for r in judge])]
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, (title, first, second) in zip(axes.flat, panels):
    for series, style in ((first, "-"), (second, "--")):
        points = [(x, y) for x, y in (series or []) if y is not None]
        if points:
            ax.plot(*zip(*points), style)
    ax.set_title(title)
plt.tight_layout()
print(RUN.name, "шагов:", len(steps), "проверок:", len(val), "оценок судьи:", len(judge))